# 14. Финальная модель: динамическая экономическая типология муниципалитетов

**Цель:** зафиксировать итоговую конфигурацию модели `resolution = 1.25`, получить динамические кластеры за 24 месяца, объединить их в 3 устойчивых экономических типа, проверить качество и устойчивость, сравнить с предыдущей конфигурацией `resolution = 1.50` и сохранить все финальные результаты.

## Финальная конфигурация

- период: 2023-01 — 2024-12;
- признаки: 5 долей потребления;
- сеть: kNN, `k = 10`;
- расстояние: Euclidean;
- вес ребра: RBF;
- алгоритм кластеризации сети: Louvain;
- `resolution = 1.25`;
- `seed = 42`;
- число устойчивых экономических типов: `K = 3`.

Все пути строятся от корня проекта, поэтому ноутбук не зависит от того, из какой директории VS Code запущен.


In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import networkx as nx

from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    calinski_harabasz_score,
    adjusted_rand_score
)

print("Библиотеки загружены.")


Библиотеки загружены.


In [ ]:
cwd = Path.cwd()
project_root = None

for path in [cwd, *cwd.parents]:
    if (path / "data" / "processed").exists():
        project_root = path
        break

if project_root is None:
    raise FileNotFoundError(
        "Не удалось найти корень проекта с папкой data/processed."
    )

processed_dir = project_root / "data" / "processed"

print("Корень проекта:", project_root)
print("Папка processed:", processed_dir)


## 1. Конфигурация финальной модели

Численные параметры здесь должны совпадать с конфигурацией, которую мы обосновали в `12_icvi_evaluation` и `13_method_comparison`.


In [3]:
FINAL_RESOLUTION = 1.25
K_NEIGHBORS = 10
RANDOM_SEED = 42
FINAL_K = 3

FEATURE_COLS = [
    "food_share",
    "health_share",
    "marketplaces_share",
    "catering_share",
    "transport_share"
]

TYPE_NAMES = {
    0: "Маркетплейсно-ориентированный",
    1: "Диверсифицированный сервисный",
    2: "Продовольственный"
}

print("resolution:", FINAL_RESOLUTION)
print("k:", K_NEIGHBORS)
print("seed:", RANDOM_SEED)
print("K экономических типов:", FINAL_K)


resolution: 1.25
k: 10
seed: 42
K экономических типов: 3


## 2. Загрузка данных

Финальная модель **не перестраивает сеть заново**. Она использует уже рассчитанные в `04_similarity_network` ежемесячные рёбра из `edges_all_months.parquet`, а затем повторно выполняет Louvain с финальным `resolution = 1.25`.

Старая типология загружается явно, чтобы сравнение с `resolution = 1.50` не зависело от содержимого памяти Jupyter.


In [4]:
features_basic = pd.read_parquet(
    processed_dir / "features_basic.parquet"
)

edges_all = pd.read_parquet(
    processed_dir / "edges_all_months.parquet"
)

old_types = (
    pd.read_parquet(
        processed_dir / "municipality_economic_types.parquet"
    )
    .rename(
        columns={
            "economic_type": "economic_type_old"
        }
    )
)

print("features_basic:", features_basic.shape)
print("edges_all:", edges_all.shape)
print("old_types:", old_types.shape)

print(
    "Период:",
    features_basic["date"].min(),
    "—",
    features_basic["date"].max()
)


features_basic: (50521, 14)
edges_all: (331489, 4)
old_types: (50521, 4)
Период: 2023-01 — 2024-12


In [5]:
for df in [features_basic, edges_all, old_types]:
    df["date"] = pd.to_datetime(df["date"])

months = sorted(features_basic["date"].unique())

print("Количество месяцев:", len(months))
print("Первый месяц:", months[0])
print("Последний месяц:", months[-1])


Количество месяцев: 24
Первый месяц: 2023-01-01 00:00:00
Последний месяц: 2024-12-01 00:00:00


In [6]:
assert set(FEATURE_COLS).issubset(features_basic.columns)

assert len(months) == 24

assert (
    features_basic[
        ["territory_id", "date"]
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    old_types[
        ["territory_id", "date"]
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    edges_all[
        ["source", "target", "date"]
    ]
    .duplicated()
    .sum()
    == 0
)

print("Базовые проверки пройдены.")


Базовые проверки пройдены.


## 3. Финальная динамическая кластеризация

Для каждого месяца:

1. берем муниципалитеты и соответствующие им ребра;
2. строим взвешенный граф;
3. применяем Louvain с `resolution = 1.25` и `seed = 42`;
4. сохраняем кластер для каждого municipality-month;
5. сохраняем число кластеров и modularity.

Важно: номера месячных кластеров не трактуются как постоянные экономические типы.


In [7]:
final_cluster_rows = []
final_monthly_summary_rows = []

for date in months:
    month_features = (
        features_basic[
            features_basic["date"] == date
        ]
        .copy()
        .reset_index(drop=True)
    )

    month_edges = (
        edges_all[
            edges_all["date"] == date
        ]
        .copy()
    )

    G = nx.Graph()

    node_ids = (
        month_features["territory_id"]
        .astype(int)
        .tolist()
    )

    G.add_nodes_from(node_ids)

    for row in month_edges.itertuples(index=False):
        G.add_edge(
            int(row.source),
            int(row.target),
            weight=float(row.weight)
        )

    communities = nx.community.louvain_communities(
        G,
        weight="weight",
        resolution=FINAL_RESOLUTION,
        seed=RANDOM_SEED
    )

    for cluster_id, community in enumerate(communities):
        for territory_id in community:
            final_cluster_rows.append({
                "territory_id": int(territory_id),
                "date": date,
                "cluster": int(cluster_id)
            })

    modularity = nx.community.modularity(
        G,
        communities,
        weight="weight"
    )

    final_monthly_summary_rows.append({
        "date": date,
        "n_municipalities": len(node_ids),
        "n_clusters": len(communities),
        "modularity": modularity,
        "n_edges": G.number_of_edges(),
        "n_components": nx.number_connected_components(G)
    })

final_clusters = pd.DataFrame(final_cluster_rows)
final_monthly_summary = pd.DataFrame(final_monthly_summary_rows)

print("Назначений:", len(final_clusters))
print("Месячных строк:", len(final_monthly_summary))

monthly_summary_display = final_monthly_summary[
    [
        "date",
        "n_municipalities",
        "n_clusters",
        "modularity",
        "n_edges",
        "n_components"
    ]
].copy()

numeric_cols = monthly_summary_display.select_dtypes(include="number").columns
monthly_summary_display[numeric_cols] = monthly_summary_display[numeric_cols].round(4)

display(monthly_summary_display)


Назначений: 50521
Месячных строк: 24


/var/folders/xz/bmnn3v0j3ys0kw7y6r2sh6fc0000gn/T/ipykernel_20684/2587563752.py:83: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].round(4)


,date,n_municipalities,n_clusters,modularity,n_edges,n_components
0,2023-01-01,2143,20,0.7914,14016,1
1,2023-02-01,2141,20,0.7900,14100,1
2,2023-03-01,2149,18,0.7949,14065,1
3,2023-04-01,2129,19,0.8015,13975,1
4,2023-05-01,2121,20,0.8077,13926,1
5,2023-06-01,2119,20,0.8078,13927,1
6,2023-07-01,2126,20,0.8112,13966,1
7,2023-08-01,2118,20,0.8025,13859,1
8,2023-09-01,2109,18,0.8045,13888,1
9,2023-10-01,2103,19,0.8067,13754,1


In [8]:
assert len(final_clusters) == len(features_basic)

assert (
    final_clusters[
        ["territory_id", "date"]
    ]
    .duplicated()
    .sum()
    == 0
)

assert final_clusters["cluster"].notna().all()

assert (
    final_monthly_summary["n_components"] == 1
).all()

print("Проверки динамической кластеризации пройдены.")


Проверки динамической кластеризации пройдены.


In [9]:
final_clusters.to_parquet(
    processed_dir / "final_clusters_resolution_125.parquet",
    index=False
)

final_monthly_summary.to_csv(
    processed_dir / "final_monthly_summary_resolution_125.csv",
    index=False
)

print("Месячные результаты сохранены.")


Месячные результаты сохранены.


## 4. Профили месячных кластеров

Для каждого месячного кластера рассчитываем средние значения пяти экономических долей и размер кластера.


In [10]:
final_cluster_profiles = (
    features_basic[
        [
            "territory_id",
            "date",
            *FEATURE_COLS
        ]
    ]
    .merge(
        final_clusters,
        on=[
            "territory_id",
            "date"
        ],
        how="inner"
    )
    .groupby(
        ["date", "cluster"],
        as_index=False
    )
    .agg(
        food_share=("food_share", "mean"),
        health_share=("health_share", "mean"),
        marketplaces_share=("marketplaces_share", "mean"),
        catering_share=("catering_share", "mean"),
        transport_share=("transport_share", "mean"),
        n_municipalities=("territory_id", "nunique")
    )
)

print(
    "Количество месячных кластеров:",
    len(final_cluster_profiles)
)

display(final_cluster_profiles.head())


Количество месячных кластеров: 455


,date,cluster,food_share,health_share,marketplaces_share,catering_share,transport_share,n_municipalities
0,2023-01-01,0,0.650021,0.087400,0.096747,0.059354,0.106479,173
1,2023-01-01,1,0.714487,0.072052,0.109116,0.030464,0.073881,160
2,2023-01-01,2,0.683537,0.074515,0.107521,0.051113,0.083314,49
3,2023-01-01,3,0.632223,0.076078,0.118356,0.043580,0.129764,98
4,2023-01-01,4,0.680799,0.078510,0.115511,0.025312,0.099867,94


In [11]:
final_cluster_profiles.to_csv(
    processed_dir / "final_cluster_profiles_resolution_125.csv",
    index=False
)

print("Профили месячных кластеров сохранены.")


Профили месячных кластеров сохранены.


## 5. Определение числа устойчивых экономических типов

Кластеризуем **профили месячных кластеров**, а не отдельные муниципалитеты.

Для учёта размера месячного кластера используется `sample_weight = n_municipalities`.

Проверяем `K = 3 ... 10`.


In [12]:
PROFILE_COLS = FEATURE_COLS.copy()

X_profiles = (
    final_cluster_profiles[
        PROFILE_COLS
    ]
    .to_numpy()
)

cluster_weights = (
    final_cluster_profiles[
        "n_municipalities"
    ]
    .to_numpy()
)

k_validation_results = []

for k in range(3, 11):
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_SEED,
        n_init=50
    )

    model.fit(
        X_profiles,
        sample_weight=cluster_weights
    )

    labels_k = model.labels_

    silhouette = silhouette_score(
        X_profiles,
        labels_k,
        metric="euclidean"
    )

    silhouette_values = silhouette_samples(
        X_profiles,
        labels_k,
        metric="euclidean"
    )

    weighted_silhouette = np.average(
        silhouette_values,
        weights=cluster_weights
    )

    calinski = calinski_harabasz_score(
        X_profiles,
        labels_k
    )

    k_validation_results.append({
        "k": k,
        "silhouette": silhouette,
        "weighted_silhouette": weighted_silhouette,
        "calinski_harabasz": calinski
    })

k_validation_df = pd.DataFrame(
    k_validation_results
)

display(k_validation_df.round(4))


,k,silhouette,weighted_silhouette,calinski_harabasz
0,3,0.3930,0.3979,422.0317
1,4,0.3439,0.3482,379.1138
2,5,0.3383,0.3442,356.9709
3,6,0.3179,0.3249,336.6940
4,7,0.3376,0.3358,377.9824
5,8,0.3148,0.3148,355.2565
6,9,0.3108,0.3105,358.7695
7,10,0.3042,0.3042,359.9629


In [13]:
k_validation_df.to_csv(
    processed_dir / "economic_type_k_validation_final_125.csv",
    index=False
)

print("Проверка K сохранена.")


Проверка K сохранена.


## 6. Финальная кластеризация профилей: K = 3

После проверки фиксируем `K = 3`.

Числовые метки KMeans (`0`, `1`, `2`) не имеют самостоятельного смысла. Поэтому после кластеризации они автоматически сопоставляются с экономической интерпретацией по профилям.


In [14]:
kmeans_final = KMeans(
    n_clusters=FINAL_K,
    random_state=RANDOM_SEED,
    n_init=50
)

kmeans_final.fit(
    X_profiles,
    sample_weight=cluster_weights
)

final_cluster_profiles["economic_type_raw"] = (
    kmeans_final.labels_
)

print(
    "Количество raw-типов:",
    final_cluster_profiles[
        "economic_type_raw"
    ].nunique()
)


Количество raw-типов: 3


In [15]:
raw_profile_rows = []

for raw_type, group in (
    final_cluster_profiles
    .groupby("economic_type_raw")
):
    row = {
        "economic_type_raw": int(raw_type)
    }

    for col in PROFILE_COLS:
        row[col] = np.average(
            group[col],
            weights=group["n_municipalities"]
        )

    raw_profile_rows.append(row)

raw_profiles = (
    pd.DataFrame(raw_profile_rows)
    .set_index("economic_type_raw")
    .sort_index()
)

display(raw_profiles.round(4))


,food_share,health_share,marketplaces_share,catering_share,transport_share
economic_type_raw,,,,,
0,0.6003,0.0636,0.2304,0.0333,0.0724
1,0.6795,0.0654,0.1420,0.0352,0.0779
2,0.5032,0.0841,0.1980,0.1040,0.1108


In [16]:
marketplace_cluster = int(
    raw_profiles["marketplaces_share"].idxmax()
)

food_cluster = int(
    raw_profiles["food_share"].idxmax()
)

remaining_clusters = [
    int(x)
    for x in raw_profiles.index
    if int(x) not in [
        marketplace_cluster,
        food_cluster
    ]
]

assert len(remaining_clusters) == 1

diversified_cluster = remaining_clusters[0]

type_mapping = {
    marketplace_cluster: 0,
    diversified_cluster: 1,
    food_cluster: 2
}

assert len(type_mapping) == FINAL_K
assert len(set(type_mapping.values())) == FINAL_K

print("Сопоставление raw → semantic:", type_mapping)


Сопоставление raw → semantic: {0: 0, 2: 1, 1: 2}


In [17]:
final_cluster_profiles["economic_type"] = (
    final_cluster_profiles[
        "economic_type_raw"
    ].map(type_mapping)
)

final_cluster_profiles[
    "economic_type_name"
] = (
    final_cluster_profiles[
        "economic_type"
    ].map(TYPE_NAMES)
)

assert final_cluster_profiles[
    "economic_type"
].notna().all()

display(
    final_cluster_profiles[
        [
            "date",
            "cluster",
            "economic_type",
            "economic_type_name",
            "n_municipalities"
        ]
    ].head()
)


,date,cluster,economic_type,economic_type_name,n_municipalities
0,2023-01-01,0,2,Продовольственный,173
1,2023-01-01,1,2,Продовольственный,160
2,2023-01-01,2,2,Продовольственный,49
3,2023-01-01,3,2,Продовольственный,98
4,2023-01-01,4,2,Продовольственный,94


## 7. Финальная типология муниципалитетов

Каждому municipality-month присваиваем устойчивый экономический тип через его месячный Louvain-кластер.


In [18]:
final_municipality_types = (
    final_clusters[
        [
            "territory_id",
            "date",
            "cluster"
        ]
    ]
    .merge(
        final_cluster_profiles[
            [
                "date",
                "cluster",
                "economic_type",
                "economic_type_name"
            ]
        ],
        on=[
            "date",
            "cluster"
        ],
        how="left"
    )
)

assert (
    final_municipality_types[
        "economic_type"
    ].notna().all()
)

assert (
    final_municipality_types[
        ["territory_id", "date"]
    ]
    .duplicated()
    .sum()
    == 0
)

print(
    "Размер финальной типологии:",
    final_municipality_types.shape
)

print(
    "Пропусков:",
    final_municipality_types["economic_type"].isna().sum()
)

display(final_municipality_types.head())


Размер финальной типологии: (50521, 5)
Пропусков: 0


,territory_id,date,cluster,economic_type,economic_type_name
0,23,2023-01-01,0,2,Продовольственный
1,30,2023-01-01,0,2,Продовольственный
2,31,2023-01-01,0,2,Продовольственный
3,32,2023-01-01,0,2,Продовольственный
4,33,2023-01-01,0,2,Продовольственный


## 8. Финальные профили трех экономических типов

Здесь профили рассчитываются уже непосредственно по всем `municipality-month` наблюдениям, поэтому каждый муниципалитет-месяц имеет одинаковый вес.


In [19]:
final_eval = (
    final_municipality_types
    .merge(
        features_basic[
            [
                "territory_id",
                "date",
                *FEATURE_COLS
            ]
        ],
        on=[
            "territory_id",
            "date"
        ],
        how="inner"
    )
)

assert len(final_eval) == len(final_municipality_types)

final_type_profiles = (
    final_eval
    .groupby(
        "economic_type"
    )[FEATURE_COLS]
    .mean()
    .sort_index()
)

final_type_profiles.index = [
    TYPE_NAMES[int(i)]
    for i in final_type_profiles.index
]

print("Финальные профили:")
display(final_type_profiles.round(4))


Финальные профили:


,food_share,health_share,marketplaces_share,catering_share,transport_share
Маркетплейсно-ориентированный,0.6003,0.0636,0.2304,0.0333,0.0724
Диверсифицированный сервисный,0.5032,0.0841,0.1980,0.1040,0.1108
Продовольственный,0.6795,0.0654,0.1420,0.0352,0.0779


In [20]:
final_type_sizes = (
    final_eval
    .groupby(
        ["economic_type", "economic_type_name"]
    )
    .size()
    .reset_index(
        name="n_observations"
    )
)

final_type_sizes["share"] = (
    final_type_sizes["n_observations"]
    / len(final_eval)
)

display(
    final_type_sizes.sort_values(
        "economic_type"
    ).round(4)
)


,economic_type,economic_type_name,n_observations,share
0,0,Маркетплейсно-ориентированный,21245,0.4205
1,1,Диверсифицированный сервисный,10360,0.2051
2,2,Продовольственный,18916,0.3744


In [21]:
final_type_profiles.to_csv(
    processed_dir / "economic_type_profiles_final_125.csv"
)

final_type_sizes.to_csv(
    processed_dir / "economic_type_sizes_final_125.csv",
    index=False
)

print("Профили и размеры типов сохранены.")


Профили и размеры типов сохранены.


## 9. Сравнение финальной типологии с прежней моделью `resolution = 1.50`

Проверяем:

- покрытие общей выборки;
- точное совпадение семантических меток;
- ARI.

ARI не зависит от конкретных числовых названий кластеров и показывает согласованность двух разбиений.


In [22]:
old_types["date"] = pd.to_datetime(
    old_types["date"]
)

final_municipality_types["date"] = pd.to_datetime(
    final_municipality_types["date"]
)

comparison_final = (
    final_municipality_types
    .merge(
        old_types[
            [
                "territory_id",
                "date",
                "economic_type_old"
            ]
        ],
        on=[
            "territory_id",
            "date"
        ],
        how="inner"
    )
)

comparison_coverage = (
    len(comparison_final)
    / len(final_municipality_types)
)

exact_match_rate = (
    comparison_final["economic_type"]
    == comparison_final["economic_type_old"]
).mean()

ari_final_vs_old = adjusted_rand_score(
    comparison_final["economic_type_old"],
    comparison_final["economic_type"]
)

comparison_metrics = pd.DataFrame({
    "metric": [
        "comparison_observations",
        "comparison_coverage",
        "exact_match_rate",
        "ARI_vs_resolution_1_50"
    ],
    "value": [
        len(comparison_final),
        comparison_coverage,
        exact_match_rate,
        ari_final_vs_old
    ]
})

display(comparison_metrics.round(6))


,metric,value
0,comparison_observations,50521.000000
1,comparison_coverage,1.000000
2,exact_match_rate,0.958493
3,ARI_vs_resolution_1_50,0.872312


**Важно:** `exact_match_rate` — это доля буквальных совпадений семантических меток на общей части выборки. Это не то же самое, что покрытие сравнения.


## 10. Финальное качество типологии

Оценка выполняется на всех `50 521` municipality-month наблюдениях по исходным пяти экономическим долям.


In [23]:
X_final = final_eval[FEATURE_COLS]
y_final = final_eval["economic_type"]

silhouette_final = silhouette_score(
    X_final,
    y_final,
    metric="euclidean"
)

calinski_final = calinski_harabasz_score(
    X_final,
    y_final
)

print(
    f"Silhouette: {silhouette_final:.4f}"
)

print(
    f"Calinski-Harabasz: {calinski_final:.2f}"
)

print(
    f"Наблюдений: {len(final_eval):,}"
)

print(
    f"Типов: {y_final.nunique()}"
)


Silhouette: 0.3456
Calinski-Harabasz: 38527.26
Наблюдений: 50,521
Типов: 3


## 11. Динамика долей экономических типов

Получаем долю муниципалитетов каждого типа в каждом месяце.


In [24]:
monthly_type_shares = (
    final_municipality_types
    .groupby(
        ["date", "economic_type", "economic_type_name"]
    )
    .size()
    .reset_index(
        name="n_municipalities"
    )
)

monthly_totals = (
    final_municipality_types
    .groupby("date")
    .size()
    .rename("total_municipalities")
)

monthly_type_shares = (
    monthly_type_shares
    .merge(
        monthly_totals,
        on="date",
        how="left"
    )
)

monthly_type_shares["share"] = (
    monthly_type_shares["n_municipalities"]
    / monthly_type_shares["total_municipalities"]
)

monthly_type_share_pivot = (
    monthly_type_shares
    .pivot(
        index="date",
        columns="economic_type_name",
        values="share"
    )
    .reset_index()
)

monthly_type_share_display = monthly_type_share_pivot.copy()

numeric_cols = monthly_type_share_display.select_dtypes(include="number").columns
monthly_type_share_display[numeric_cols] = monthly_type_share_display[numeric_cols].round(4)

display(monthly_type_share_display)


/var/folders/xz/bmnn3v0j3ys0kw7y6r2sh6fc0000gn/T/ipykernel_20684/4135790697.py:44: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  monthly_type_share_pivot.round(4)


economic_type_name,date,Диверсифицированный сервисный,Маркетплейсно-ориентированный,Продовольственный
0,2023-01-01,0.1778,0.0826,0.7396
1,2023-02-01,0.1709,0.1579,0.6712
2,2023-03-01,0.1810,0.2224,0.5966
3,2023-04-01,0.1221,0.1419,0.7360
4,2023-05-01,0.1273,0.1315,0.7412
5,2023-06-01,0.1265,0.1642,0.7093
6,2023-07-01,0.2173,0.1839,0.5988
7,2023-08-01,0.2276,0.2262,0.5463
8,2023-09-01,0.2404,0.2608,0.4988
9,2023-10-01,0.2192,0.4280,0.3528


In [25]:
monthly_type_shares.to_csv(
    processed_dir / "monthly_economic_type_shares_final_125_long.csv",
    index=False
)

monthly_type_share_pivot.to_csv(
    processed_dir / "monthly_economic_type_shares_final_125.csv",
    index=False
)

print("Динамика долей сохранена.")


Динамика долей сохранена.


## 12. Переходы между экономическими типами

Для каждого соседнего месяца считаем, сколько муниципалитетов осталось в том же типе и сколько перешло в другой.


In [26]:
transition_rows = []

date_list = months

for i in range(len(date_list) - 1):
    date_from = date_list[i]
    date_to = date_list[i + 1]

    a = (
        final_municipality_types[
            final_municipality_types["date"] == date_from
        ][
            ["territory_id", "economic_type", "economic_type_name"]
        ]
        .rename(
            columns={
                "economic_type": "type_from",
                "economic_type_name": "type_from_name"
            }
        )
    )

    b = (
        final_municipality_types[
            final_municipality_types["date"] == date_to
        ][
            ["territory_id", "economic_type", "economic_type_name"]
        ]
        .rename(
            columns={
                "economic_type": "type_to",
                "economic_type_name": "type_to_name"
            }
        )
    )

    merged = a.merge(
        b,
        on="territory_id",
        how="inner"
    )

    flow = (
        merged
        .groupby(
            [
                "type_from",
                "type_from_name",
                "type_to",
                "type_to_name"
            ]
        )
        .size()
        .reset_index(
            name="n_municipalities"
        )
    )

    flow["date_from"] = date_from
    flow["date_to"] = date_to

    transition_rows.append(flow)

type_transitions = pd.concat(
    transition_rows,
    ignore_index=True
)

display(type_transitions.head(12))


,type_from,type_from_name,type_to,type_to_name,n_municipalities,date_from,date_to
0,0,Маркетплейсно-ориентированный,0,Маркетплейсно-ориентированный,166,2023-01-01,2023-02-01
1,0,Маркетплейсно-ориентированный,1,Диверсифицированный сервисный,3,2023-01-01,2023-02-01
2,0,Маркетплейсно-ориентированный,2,Продовольственный,8,2023-01-01,2023-02-01
3,1,Диверсифицированный сервисный,0,Маркетплейсно-ориентированный,17,2023-01-01,2023-02-01
4,1,Диверсифицированный сервисный,1,Диверсифицированный сервисный,356,2023-01-01,2023-02-01
5,1,Диверсифицированный сервисный,2,Продовольственный,7,2023-01-01,2023-02-01
6,2,Продовольственный,0,Маркетплейсно-ориентированный,155,2023-01-01,2023-02-01
7,2,Продовольственный,1,Диверсифицированный сервисный,7,2023-01-01,2023-02-01
8,2,Продовольственный,2,Продовольственный,1421,2023-01-01,2023-02-01
9,0,Маркетплейсно-ориентированный,0,Маркетплейсно-ориентированный,256,2023-02-01,2023-03-01


In [27]:
type_transitions.to_csv(
    processed_dir / "economic_type_transitions_final_125.csv",
    index=False
)

print("Переходы между типами сохранены.")


Переходы между типами сохранены.


## 13. Проверка стабильности `resolution = 1.25`

Основная многократная проверка устойчивости уже находится в `12_icvi_evaluation`. Здесь только подхватываем сохраненный результат и выводим строку для финальной конфигурации.


In [ ]:
stability_path = (
    processed_dir
    / "louvain_stability_dec2024.csv"
)

if stability_path.exists():
    stability_df = pd.read_csv(
        stability_path
    )

    final_stability = stability_df[
        stability_df["resolution"] == FINAL_RESOLUTION
    ]

    display(final_stability)

else:
    print(
        "Файл stability не найден. "
        "Сначала выполните 12_icvi_evaluation.ipynb."
    )


,resolution,ARI_mean,ARI_min,ARI_max
3,1.25,0.685564,0.601698,0.746273


## 14. Итоговая таблица финальной модели


In [29]:
final_evaluation = pd.DataFrame({
    "metric": [
        "resolution",
        "k_neighbors",
        "seed",
        "n_types",
        "n_observations",
        "n_months",
        "silhouette",
        "calinski_harabasz",
        "ARI_vs_resolution_1_50",
        "comparison_coverage",
        "exact_match_rate_vs_resolution_1_50",
        "mean_monthly_modularity",
        "min_monthly_clusters",
        "max_monthly_clusters"
    ],
    "value": [
        FINAL_RESOLUTION,
        K_NEIGHBORS,
        RANDOM_SEED,
        int(y_final.nunique()),
        int(len(final_eval)),
        int(len(months)),
        float(silhouette_final),
        float(calinski_final),
        float(ari_final_vs_old),
        float(comparison_coverage),
        float(exact_match_rate),
        float(final_monthly_summary["modularity"].mean()),
        int(final_monthly_summary["n_clusters"].min()),
        int(final_monthly_summary["n_clusters"].max())
    ]
})

display(final_evaluation)


,metric,value
0,resolution,1.250000
1,k_neighbors,10.000000
2,seed,42.000000
3,n_types,3.000000
4,n_observations,50521.000000
5,n_months,24.000000
6,silhouette,0.345623
7,calinski_harabasz,38527.262639
8,ARI_vs_resolution_1_50,0.872312
9,comparison_coverage,1.000000


In [30]:
final_municipality_types.to_parquet(
    processed_dir
    / "municipality_economic_types_final.parquet",
    index=False
)

final_evaluation.to_csv(
    processed_dir
    / "final_model_evaluation.csv",
    index=False
)

print("======================================")
print("Все финальные артефакты сохранены.")
print("======================================")

for filename in [
    "final_clusters_resolution_125.parquet",
    "final_monthly_summary_resolution_125.csv",
    "final_cluster_profiles_resolution_125.csv",
    "municipality_economic_types_final.parquet",
    "economic_type_profiles_final_125.csv",
    "economic_type_sizes_final_125.csv",
    "economic_type_k_validation_final_125.csv",
    "monthly_economic_type_shares_final_125.csv",
    "monthly_economic_type_shares_final_125_long.csv",
    "economic_type_transitions_final_125.csv",
    "final_model_evaluation.csv"
]:
    path = processed_dir / filename
    print(
        f"{filename}:",
        "OK" if path.exists() else "NOT FOUND"
    )


Все финальные артефакты сохранены.
final_clusters_resolution_125.parquet: OK
final_monthly_summary_resolution_125.csv: OK
final_cluster_profiles_resolution_125.csv: OK
municipality_economic_types_final.parquet: OK
economic_type_profiles_final_125.csv: OK
economic_type_sizes_final_125.csv: OK
economic_type_k_validation_final_125.csv: OK
monthly_economic_type_shares_final_125.csv: OK
monthly_economic_type_shares_final_125_long.csv: OK
economic_type_transitions_final_125.csv: OK
final_model_evaluation.csv: OK


## Финальный контроль

Перед переходом к `15_final_visualization.ipynb` должны выполняться следующие условия:

- 24 месяца;
- 50 521 municipality-month наблюдение;
- 3 экономических типа;
- нет пропусков в `economic_type`;
- `resolution = 1.25`;
- `k = 10`;
- `seed = 42`;
- финальные профили трех типов сохранены;
- ARI и сравнение с `resolution = 1.50` рассчитаны;
- финальные файлы присутствуют в `data/processed/`.
